In [ ]:
import pandas as pd
import numpy as np
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path
from typing import Optional
import matplotlib.pyplot as plt
import polars as pl
import psutil
import seaborn as sns

# 1. Coinmetrics

In [ ]:
# read data
df = pd.read_csv("../data/Coin Metrics/coinmetrics_btc.csv")
df["time"] = pd.to_datetime(df["time"])
df.head()

In [ ]:
# test the commit
start = "2018-01-01"
end = "2025-12-31"
df = df[(df["time"] >= start) & (df["time"] <= end)].copy()
# sort and set the index
df = df.sort_values("time").set_index("time")

In [ ]:
# check missing date
all_days = pd.date_range(start, end, freq="D")
missing_days = all_days.difference(df.index)
len(missing_days)

In [ ]:
# missing rate
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("missing rate Top 20（%）：")
print(missing_pct.head(20))

key_cols = [
    "PriceUSD", "CapMrktCurUSD", "HashRate", "TxCnt",
    "CapMVRVCur", "SplyCur", "FeeTotNtv",
    "FlowInExUSD", "FlowOutExUSD",
    "volume_reported_spot_usd_1d"
]
key_cols = [c for c in key_cols if c in df.columns]

print("\nkey row describe：")
print(df[key_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

In [ ]:
# 6) create metrics

px = df["PriceUSD"]

df["ret_1d"] = px.pct_change()
df["logret_1d"] = np.log(px).diff()

# annualized vol
df["vol_30d"] = df["logret_1d"].rolling(30).std() * np.sqrt(365)

# 200 DMA
df["ma_200"] = px.rolling(200).mean()
df["price_vs_ma200"] = px / df["ma_200"] - 1

# Drawdown
rolling_max = px.cummax()
df["drawdown"] = px / rolling_max - 1


In [ ]:
df.columns

In [ ]:
df.head()

# 2. Coinmetrics EDA

In [ ]:
metrics = ["PriceUSD", "CapMrktCurUSD", "HashRate"]
df[metrics].describe()

In [ ]:
# Price Trend
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["PriceUSD"])
plt.title("BTC Price — 2018-01-01 to 2025-12-31")
plt.xlabel("Date")
plt.ylabel("PriceUSD")
plt.tight_layout()
plt.show()

In [ ]:
# BTC Price (Log Scale) with 200-Day Moving Average
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["PriceUSD"], label="BTC Price (USD)")
plt.plot(df.index, df["ma_200"], label="200-Day Moving Average", linestyle="--")

plt.yscale("log")

plt.title("BTC Price (Log Scale) with 200-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Price (USD, Log Scale)")

plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Daily Profit Distribution
rets = df["ret_1d"].dropna()
plt.figure(figsize=(10, 4))
plt.hist(rets, bins=200)
plt.title("Daily Returns Histogram")
plt.xlabel("Daily Return")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
print("\nWorst 10 Days (ret_1d):")
print(rets.nsmallest(10))

print("\nBest 10 Days (ret_1d):")
print(rets.nlargest(10))

In [ ]:
# 30D Rolling Annualized Volatility
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["vol_30d"])
plt.title("30D Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown Plot
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["drawdown"])
plt.title("Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.tight_layout()
plt.show()

In [ ]:
#Correlation Heatmap
correlation_cols = ["PriceUSD", "CapMrktCurUSD", "HashRate", "TxCnt"]
corr_df = df[correlation_cols].apply(pd.to_numeric, errors="coerce")

# corr
corr = corr_df.corr()

# heatmap
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="viridis", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
corr_cols = [
    "PriceUSD", "CapMrktCurUSD", "HashRate", "TxCnt", "TxTfrCnt",
    "CapMVRVCur", "SplyCur", "AdrActCnt", "AdrBalCnt",
    "FeeTotNtv", "FlowInExUSD", "FlowOutExUSD",
    "volume_reported_spot_usd_1d"
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="viridis", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Build accumulation common metrics
# net inflow >0 means net outflow ，<0 means net inflow
df["net_flow_usd"] = df["FlowOutExUSD"] - df["FlowInExUSD"]

# PnL in next 30 days
h = 30
df["fwd_ret_30d"] = df["PriceUSD"].shift(-h) / df["PriceUSD"] - 1

# Forward vol in 30D log return
logret = np.log(df["PriceUSD"]).diff()
df["fwd_vol_30d"] = logret.shift(-1).rolling(h).std() * np.sqrt(365)

# Max Drawdown in next 30D
# Make the lowest price in the forward window, then convert to max drawdown
future_min_price = df["PriceUSD"].shift(-1).rolling(h).min()
df["fwd_min_dd_30d"] = future_min_price / df["PriceUSD"] - 1 

# Remove missing data in the last 30D
eda = df.dropna(subset=["fwd_ret_30d", "fwd_vol_30d", "fwd_min_dd_30d"]).copy()


In [ ]:
#One dimensional bin analysis 

#Group into 5 buckets by MVRV
eda["mvrv_q5"] = pd.qcut(eda["CapMVRVCur"], 5, labels=[1,2,3,4,5])
mvrv_table = eda.groupby("mvrv_q5").agg(
    n=("PriceUSD", "size"),
    avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
    med_fwd_ret_30d=("fwd_ret_30d", "median"),
    avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
    prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
    prob_dd_lt_30pct=("fwd_min_dd_30d", lambda x: (x < -0.30).mean()),
).reset_index()

print("\n===  MVRV（CapMVRVCur）5 Buckets Grouping Result ===")
print(mvrv_table)

plt.figure(figsize=(10,4))
plt.plot(mvrv_table["mvrv_q5"], mvrv_table["avg_fwd_ret_30d"], marker="o")
plt.title("MVRV Quintiles vs Future 30-Day Average Returns")
plt.xlabel("MVRV Quintile (1=Undervalued, 5=Overvalued)")
plt.ylabel("Average fwd_ret_30d")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Based on net_flow_usd group into 5
eda["netflow_q5"] = pd.qcut(eda["net_flow_usd"], 5, labels=[1,2,3,4,5])
netflow_table = eda.groupby("netflow_q5").agg(
    n=("PriceUSD", "size"),
    avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
    med_fwd_ret_30d=("fwd_ret_30d", "median"),
    avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
    prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
    prob_dd_lt_30pct=("fwd_min_dd_30d", lambda x: (x < -0.30).mean()),
).reset_index()

print("\n=== Net Flow (net_flow_usd) Quintile Binning Results ===")
print(netflow_table)

plt.figure(figsize=(10,4))
plt.plot(netflow_table["netflow_q5"], netflow_table["prob_dd_lt_20pct"], marker="o")
plt.title("Net Flow Quintiles vs Future 30-Day Drawdown <-20% Probability")
plt.xlabel("Net Flow Quintile (1=Strong Inflow, 5=Strong Outflow)")
plt.ylabel("P(fwd_min_dd_30d < -20%)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#  AdrActCnt 5 binning
eda["adract_q5"] = pd.qcut(eda["AdrActCnt"], 5, labels=[1,2,3,4,5])
adr_table = eda.groupby("adract_q5").agg(
    n=("PriceUSD", "size"),
    avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
    med_fwd_ret_30d=("fwd_ret_30d", "median"),
    avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
    prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
    prob_dd_lt_30pct=("fwd_min_dd_30d", lambda x: (x < -0.30).mean()),
).reset_index()

print("\n=== （AdrActCnt）Quintile Binning Results ===")
print(adr_table)

plt.figure(figsize=(10,4))
plt.plot(adr_table["adract_q5"], adr_table["avg_fwd_vol_30d"], marker="o")
plt.title("AdrActCnt Quintiles vs Future 30-Day Average Volatility")
plt.xlabel("AdrActCnt Quintile (1=Low Activity, 5=High Activity)）")
plt.ylabel("Average fwd_vol_30d（Annualized）")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 2D Binning: MVRV × Netflow
pivot = eda.pivot_table(
    index="mvrv_q5",
    columns="netflow_q5",
    values="fwd_ret_30d",
    aggfunc="mean"
)

plt.figure(figsize=(9,6))
sns.heatmap(pivot, annot=True, cmap="viridis", fmt=".3f")

plt.title("2D Binning: MVRV Quintile × Netflow Quintile\nAverage 30-Day Forward Return")
plt.xlabel("Netflow Quintile (1 = Net Inflow ↑ Selling Pressure, 5 = Net Outflow ↑ Accumulation)")
plt.ylabel("MVRV Quintile (1 = Undervalued, 5 = Overvalued)")

plt.tight_layout()
plt.show()

# 3. Polymarket data loading

In [ ]:
#read polymarket data
import sys
import os
import polars as pl
#from pathlib import path

sys.path.append(os.path.abspath('..'))
# Import the loader from the template provided by the project
from eda_starter_template import load_polymarket_data, POLYMARKET_DIR

# Load the dictionary of DataFrames
poly_data_dict = load_polymarket_data(POLYMARKET_DIR)

# Extract individual DataFrames for easier use
if poly_data_dict:
    df_markets = poly_data_dict.get("markets")
    df_odds = poly_data_dict.get("odds")
    df_summary = poly_data_dict.get("summary")
    
    print("Data loaded successfully!")
else:
    print("Data directory not found. Check if the 'data/Polymarket' folder exists.")

In [ ]:
tokens_path = POLYMARKET_DIR / "finance_politics_tokens.parquet"
event_path = POLYMARKET_DIR / "finance_politics_event_stats.parquet"

df_tokens = (
    pl.scan_parquet(tokens_path)
    .collect()
)
df_event = (
    pl.scan_parquet(event_path)
    .collect()
)
# trades_path = POLYMARKET_DIR / "finance_politics_trades.parquet`"
# df_trades = (
#     pl.scan_parquet(trades_path)
#     .collect()
# )

# 3.1 Markets

In [ ]:
df_markets.head()

# 3.2 Tokens

In [ ]:
df_tokens

In [ ]:
df_tokens.select("outcome").unique()

# 3.4 Odds History

In [ ]:
df_odds = df_odds.with_columns(
    pl.col("timestamp").dt.date().alias("date")
)

df_odds.head()

In [ ]:
df_tokens_simple = (
    df_tokens
    .with_columns(pl.col("outcome").str.to_lowercase().alias("outcome_lower"))
    .filter(
        pl.col("outcome_lower").is_in(["yes", "no", "up", "down"])
    )
)

In [ ]:
df_odds_simple = df_odds.join(
    df_tokens_simple.select(["market_id", "token_id", "outcome"]),
    on=["market_id", "token_id"],
    how="inner"
)

df_odds_simple.head()

In [ ]:
df_daily = (
    df_odds_simple
    .sort(["market_id", "date", "timestamp"])
    .group_by(["market_id", "date"])
    .agg([
        pl.col("price").last().alias("p_close"),
        pl.count().alias("update_count")
    ])
    .sort(["market_id", "date"])
)

df_daily = df_daily.with_columns(
    (pl.col("p_close") - pl.col("p_close").shift(1))
    .over("market_id")
    .alias("p_diff_1d")
)

df_daily = df_daily.with_columns(
    pl.col("p_diff_1d")
    .rolling_std(7)
    .over("market_id")
    .alias("p_vol_7d")
)

In [ ]:
df_daily.describe()

In [ ]:
daily_index = (
    df_daily
    .group_by("date")
    .agg([
        pl.col("p_vol_7d").mean().alias("risk_index"),
        pl.col("p_diff_1d").mean().alias("prob_shift_index"),
        pl.col("update_count").mean().alias("attention_index"),
        pl.count().alias("num_markets")
    ])
    .sort("date")
)

In [ ]:
daily_pd = pd.DataFrame(daily_index.to_dict(as_series=False))

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(daily_pd["date"], daily_pd["risk_index"])
plt.title("Polymarket Risk Index (Mean 7D Prob Vol)")
plt.xlabel("Date")
plt.ylabel("Risk Index")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(daily_pd["date"], daily_pd["attention_index"])
plt.title("Polymarket Attention Index")
plt.xlabel("Date")
plt.ylabel("Avg Update Count")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
btc_pd = df.reset_index()[["time", "PriceUSD"]]
daily_pd["date"] = pd.to_datetime(daily_pd["date"]).dt.normalize()
btc_pd["time"] = pd.to_datetime(btc_pd["time"]).dt.normalize()

merged = daily_pd.merge(
    btc_pd,
    left_on="date",
    right_on="time",
    how="inner"
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,5))

ax1.plot(merged["date"], merged["PriceUSD"], color="blue")
ax1.set_ylabel("BTC Price", color="blue")

ax2 = ax1.twinx()
ax2.plot(merged["date"], merged["risk_index"], color="red", alpha=0.6)
ax2.set_ylabel("Risk Index", color="red")

plt.title("BTC Price vs Polymarket Risk Index")
plt.tight_layout()
plt.show()

In [ ]:
#分箱分析
merged["risk_q5"] = pd.qcut(merged["risk_index"], 5, labels=[1,2,3,4,5])

risk_table = merged.groupby("risk_q5").agg(
    avg_btc_price=("PriceUSD","mean")
)

print(risk_table)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(risk_table.index, risk_table["avg_btc_price"], marker="o")
plt.title("Risk Quintile vs BTC Avg Price")
plt.xlabel("Risk Quintile")
plt.ylabel("Avg BTC Price")
plt.grid(True)
plt.show()